# 有状态聊天机器人

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
from IPython.display import Markdown, display, update_display
from openai import OpenAI

### GPT 设置

In [ ]:
# 【注】import os
# 【注】from dotenv import load_dotenv

# 【注】load_dotenv(override=True)
# 【注】api_key = os.getenv('OPENAI_API_KEY')

# 【注】if not api_key:
# 【注】print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 【注】elif not api_key.startswith("sk-proj-"):
# 【注】print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 【注】elif api_key.strip() != api_key:
# 【注】print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# 【注】else:
# 【注】print("API key found and looks good so far!")

# 【注】MODEL = "gpt-5-nano"
# 
# 【注】my_ai = OpenAI()

### Ollama 设置

In [ ]:
BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "llama3.2"

my_ai = OpenAI(base_url=BASE_URL, api_key=API_KEY)

#### 其他设置

In [ ]:
# 【注】MODEL = "deepseek-r1:1.5b"

### 易记提示词

In [ ]:
# 【注】SYSTEM_PROMPT = (
# 【注】"You are a helpless assistant that doesn't know anything. "
# 【注】"You make up answers as you go along because you are afraid of being judged. "
# 【注】"You answer with absolute confidence and a serious tone, but most of the information you provide is wrong or made up. "
# 【注】"You are having a conversation with me so your answers shouldn't be too long. "
# 【注】"Reply in markdown. Do not wrap the markdown in a code block - respond just with the markdown."
# )

# 【注】THINKING_PROMPT = "First describe your thought process then give the answer"

## 系统/用户提示词

### 模板

In [ ]:
MARKDOWN_PROMPT = "Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."

### 系统

In [ ]:
SYSTEM_PROMPT = (
    "You are a basic assistant. "
    "You solve simple tasks. "
    "Your answers are serious and straight to the point. "
    + MARKDOWN_PROMPT
)

## 辅助函数

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def display_stream(stream):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
# （代码逻辑保持原样；以下为小白向中文旁注）
def do_chat(user_prompt, messages):
    print("Q:", user_prompt)

    print("Answering...")

    response = my_ai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    full_response = display_stream(response)
    messages.append({"role": "assistant", "content": full_response})

    user_prompt = input("Wanna go further? Enter your prompt, otherwise just press enter: ")

    if user_prompt:
        messages.append({"role": "user", "content": user_prompt})
        print("Answering...")
        do_chat(user_prompt, messages)

## 正式对话

In [ ]:
user_prompt = input("Enter your first prompt: ")
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]
do_chat(user_prompt, messages)

### 调试

In [ ]:
headers = messages[0].keys()

table = "| " + " | ".join(headers) + " |\n"
table += "| " + " | ".join("---" for _ in headers) + " |\n"

for row in messages:
    table += "| " + " | ".join(str(row.get(h, "")) for h in headers) + " |\n"

display(Markdown(table))